# 164 — Explicabilidad, incertidumbre y calibración

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Auditar calibración exige comparar confianza declarada contra frecuencia real
sobre un conjunto fijo; el contrato reproducible (kind + evidence) permite recomputar el ECE y el
diagrama de fiabilidad de forma consistente entre corridas.


In [ ]:
result = run_lab("evaluation", seed=164)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
Bin [0.5,0.75): preds (0.6,0),(0.65,1),(0.7,1)   -> n=3, acc=2/3=0.667, conf=(0.6+0.65+0.7)/3=0.650
Bin [0.75,1.0]: preds (0.85,1),(0.9,0),(0.95,1)  -> n=3, acc=2/3=0.667, conf=(0.85+0.9+0.95)/3=0.900
|acc-conf|: 0.017 y 0.233
ECE = (3/6)*0.017 + (3/6)*0.233 = 0.0083 + 0.1167 = 0.125
```

El primer bin está casi calibrado; el segundo está sobreconfiado (dice 0.90, acierta 0.667).

**Ejercicio 3 (esquema).** (a) Un detector médico que dice 0.99 en casi todos los positivos y
acierta el 95 %: alta accuracy, pero la confianza 0.99 no se corresponde con 0.95 → mal calibrado.
(b) Un predictor meteorológico que dice "60 % de lluvia" y llueve el 60 % de esas veces: solo acierta
la clase el 60 %, pero sus probabilidades son fieles → bien calibrado. No se contradicen porque
accuracy mide *cuánto acierta* y calibración mide *si el número de confianza es honesto*.

**Ejercicio 4.** Eliges **SHAP** e invocas la propiedad de **aditividad/eficiencia**: los valores
de Shapley suman exactamente la diferencia entre la predicción y el valor base. Advertencia: es una
descomposición del comportamiento del *modelo*; una feature con alto aporte puede ser un proxy
correlacionado (p. ej. código postal ↔ etnia), no una causa, con riesgo de fairness.


In [ ]:
# Verificación numérica del Ejercicio 2
preds = [(0.6,0),(0.65,1),(0.7,1),(0.85,1),(0.9,0),(0.95,1)]
def bin_stats(items):
    n = len(items)
    acc = sum(a for _, a in items) / n
    conf = sum(c for c, _ in items) / n
    return n, acc, conf
b1 = [p for p in preds if p[0] < 0.75]
b2 = [p for p in preds if p[0] >= 0.75]
N = len(preds)
ece = 0.0
for b in (b1, b2):
    n, acc, conf = bin_stats(b)
    ece += (n / N) * abs(acc - conf)
    print(f"n={n} acc={acc:.3f} conf={conf:.3f} gap={abs(acc-conf):.3f}")
print(f"ECE={ece:.3f}")
assert round(ece, 3) == 0.125


## Reflexión (guía)

1. El ECE mide la brecha media (ponderada por bin) entre la confianza declarada y la frecuencia
   real de acierto; alto = las probabilidades no son fiables (típicamente sobreconfianza).
2. Porque describen cómo pesa la feature en la función aprendida, que puede haber capturado
   correlaciones o proxies del mundo, no relaciones causales.
3. La **epistémica** (ignorancia del modelo por falta de datos); la aleatoria es ruido irreducible.
